# ZuCo: reconstruct the paper’s four main figures
**ZuCo sentiment reading only.** Python/Colab reconstruction of Pfeiffer et al. (2020), [NeuroImage 116934](https://doi.org/10.1016/j.neuroimage.2020.116934).

Run all. A GPU is recommended for permutation work; filtering, regression and mixed models still use the CPU. CPU fallback works but is slower. This is a substantial new source pass, potentially several hours, not a quick redraw of the older plots.

This uses **original SST word labels and all eligible word-associated fixations**. It does not reuse our agent sentiment labels or first-fixation-only selection. It reconstructs the published figure panels, not their exact numerical results. **Blocks failing the 8-ms clock-fit limit are excluded and counted explicitly; inspect source coverage before interpreting the figures.** Corpus preprocessing is reused; timing tolerance, Python regression, matching and statistical implementation differ explicitly. See the final methods cell.

**Storage:** approximately **1.5 GB additional peak allowance** on Drive, mostly resumable permutation maps. Those maps are removed only after final statistics/figures succeed. Final reports, coefficients, event audits and E62 trial rasters are expected to occupy roughly **50–200 MB**. No second full-channel trial archive is saved. Sources are downloaded temporarily into Colab, one reader at a time. A fresh runtime needs enough temporary Colab disk/RAM as well.

Output: `Thesis/Results/word_sentiment_eeg/zuco_paper_figures/`. Completed readers are verified and skipped on rerun; an interrupted reader may need repeating. Do not run two copies into this folder simultaneously. Existing ERP/classification results remain in their existing folders.


## 1. Mount Drive and load the public code
No GitHub token is needed. Drive authentication is used to verify free space before extraction.

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures'
REPO=Path('/content/word-sentiment-eeg-paper-figures')
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
    git('-C',REPO,'pull','--ff-only','origin','main')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)


## 2. Prepare readers, run the analyses, save four figures
Defaults: 12 SR readers, 5,000 topographic-consistency permutations and 1,000 cluster permutations. No decoding grid or TeCo extraction runs here.

Logs show `PREPARE READER n/12`, source block, regression channel, permutation chunk and mixed-model progress. A heartbeat alone is not evidence that computation advanced. If interrupted, rerun this cell in the same runtime first; completed reader checkpoints are reused. Source/method fingerprint changes stop instead of mixing runs.


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/prepare_zuco_paper_figures.py'),
         '--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='ZuCo paper figures: native source preparation and inference')


## 3. Figure 1 — selection counts and raw versus cleaned responses

In [ ]:
display(Image(filename=str(OUT/'figures/figure_1_selection_cleaning.png')))


## 4. Figure 2 — raw FRPs
Channel/time heatmaps; per-trial GFP; mixed-model, cluster and topographic-consistency results; predeclared topography window.

In [ ]:
display(Image(filename=str(OUT/'figures/figure_2_raw_frp.png')))


## 5. Figure 3 — regression-cleaned FRPs
Same accepted event identities as the raw figure. All effects and significance are calculated from this run.

In [ ]:
display(Image(filename=str(OUT/'figures/figure_3_cleaned_frp.png')))


## 6. Figure 4 — control analyses
Eye velocity, neighboring fixations, word length, fixation duration and word frequency.

In [ ]:
display(Image(filename=str(OUT/'figures/figure_4_controls.png')))


## 7. Audit counts, numerical checks and method differences

In [ ]:
display(pd.read_csv(OUT/'analysis/source_coverage.csv'))
display(pd.read_csv(OUT/'analysis/exclusion_counts.csv'))
rows=[]
for p in sorted((OUT/'readers').glob('*/manifest.json')):
    v=json.loads(p.read_text());rows.append(dict(reader=v['subject'],negative=v['accepted_counts'][0],positive=v['accepted_counts'][1],paired=v['pairs']))
display(pd.DataFrame(rows))
print(json.dumps(json.loads((OUT/'analysis/analysis_manifest.json').read_text()),indent=2))
display(Markdown((OUT/'METHOD_DIFFERENCES.md').read_text()))
print('PNG and editable SVG figures:',OUT/'figures')
print(f'Current output size: {sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())/1e6:.1f} MB')


### Optional: redraw only, after a completed run
This reads the saved statistics, not the native EEG. Run this cell manually only if you need to regenerate image files. Model fitting and permutations do not repeat.


In [ ]:
# Uncomment only when you want a redraw:
# subprocess.run([sys.executable,str(REPO/'scripts/prepare_zuco_paper_figures.py'),
#                 '--thesis-root',str(THESIS),'--plots-only'],check=True)
